# LoRA and QLoRA from First Principles
## A beginner-to-deep-dive notebook with concrete matrices, worked examples, gradients, transformer scaling, and memory intuition

This notebook assumes **no prior knowledge of LoRA or QLoRA**.

We will build the ideas in this order:

1. What fine-tuning means
2. Why full fine-tuning is expensive
3. A single linear layer as a matrix
4. What a weight update looks like
5. The key LoRA idea: learn a low-rank update instead of a full update
6. A fully worked numerical LoRA example
7. Parameter-count savings
8. How gradients update only the LoRA matrices
9. Where LoRA is inserted inside a transformer
10. How one-layer intuition scales to dozens of transformer layers
11. Quantization from first principles
12. What QLoRA changes
13. A concrete QLoRA-style toy example
14. NF4, double quantization, and paged optimizers
15. Training vs inference
16. Practical implementation patterns
17. Common misconceptions
18. Final comparison: full fine-tuning vs LoRA vs QLoRA

---

### Important notation

Throughout the notebook:

- $x$ = input activation vector
- $W_0$ = pretrained model weight matrix
- $W$ = effective weight matrix used during fine-tuning
- $\Delta W$ = learned change to the pretrained matrix
- $A, B$ = LoRA trainable matrices
- $r$ = LoRA rank
- $\alpha$ = LoRA scaling hyperparameter
- $d_{in}$ = input dimension
- $d_{out}$ = output dimension

We will use the convention


$$
W_0 \in \mathbb{R}^{d_{out}\times d_{in}}
$$


and a column-vector input


$$
x\in\mathbb{R}^{d_{in}\times 1}.
$$


Then


$$
y = W_0x.
$$



> **Math rendering note:** This revised version uses `$...$` for inline mathematics and `$$...$$` for display mathematics, which is the most broadly compatible MathJax style across Jupyter Notebook, JupyterLab, VS Code notebooks, and Colab.


# 1. What does fine-tuning mean?

Suppose a pretrained language model already knows a great deal about language.

You now want to adapt it to a special task:

- answer questions in your company's style,
- understand medical terminology,
- follow a particular instruction format,
- produce SQL,
- classify legal documents,
- speak in a special tone,
- or behave better on some downstream task.

The simplest idea is **full fine-tuning**.

If the pretrained parameter tensor is $\theta_0$, training produces


$$
\theta = \theta_0 + \Delta\theta.
$$


In ordinary full fine-tuning, essentially every trainable weight in the model may change.

For one matrix:


$$
W = W_0 + \Delta W.
$$


The problem is that modern LLMs contain billions of parameters.

If a model has 7 billion parameters, full fine-tuning potentially requires:

- storage for billions of trainable weights,
- gradients for billions of weights,
- optimizer state for billions of weights,
- temporary activations needed for backpropagation.

The optimizer state can actually dominate the memory cost.

LoRA asks a very clever question:

> Do we really need to learn every element of $\Delta W$ independently?

# 2. Start with one tiny linear layer

Forget transformers for a moment.

Suppose one layer has


$$
d_{in}=3,\qquad d_{out}=4.
$$


Then the layer weight is a $4\times3$ matrix:


$$
W_0=
\begin{bmatrix}
1 & 2 & 0\\
0 & -1 & 3\\
2 & 1 & 1\\
-1 & 0 & 2
\end{bmatrix}.
$$


Take


$$
x=
\begin{bmatrix}
2\\
1\\
-1
\end{bmatrix}.
$$


The output is


$$
y=W_0x.
$$


Let's calculate it manually.

In [ ]:
import numpy as np

W0 = np.array([
    [ 1.,  2.,  0.],
    [ 0., -1.,  3.],
    [ 2.,  1.,  1.],
    [-1.,  0.,  2.]
])

x = np.array([[2.],
              [1.],
              [-1.]])

y = W0 @ x

print("W0 =\n", W0)
print("\nx =\n", x)
print("\ny = W0 @ x =\n", y)

By hand:


$$
y_1=1(2)+2(1)+0(-1)=4
$$


$$
y_2=0(2)-1(1)+3(-1)=-4
$$


$$
y_3=2(2)+1(1)+1(-1)=4
$$


$$
y_4=-1(2)+0(1)+2(-1)=-4.
$$


So


$$
y=
\begin{bmatrix}
4\\
-4\\
4\\
-4
\end{bmatrix}.
$$


This tiny matrix is conceptually the same kind of object as a huge projection matrix inside an LLM.

The real matrices are simply much larger.

# 3. What full fine-tuning would do to this layer

Suppose training decides the layer should change by


$$
\Delta W=
\begin{bmatrix}
0.10 & 0.00 & -0.10\\
0.05 & 0.20 & 0.00\\
0.00 & -0.10 & 0.15\\
0.10 & 0.05 & -0.05
\end{bmatrix}.
$$


Then


$$
W = W_0+\Delta W.
$$


Because $W_0$ contains $4\times3=12$ values, a full update $\Delta W$ also contains 12 independently trainable values.

For a real matrix of shape, say,


$$
4096\times4096,
$$


the update contains


$$
4096^2=16,777,216
$$


trainable values **for that one matrix alone**.

In [ ]:
delta_W_full = np.array([
    [0.10, 0.00, -0.10],
    [0.05, 0.20,  0.00],
    [0.00,-0.10,  0.15],
    [0.10, 0.05, -0.05]
])

W_full = W0 + delta_W_full
y_full = W_full @ x

print("Full fine-tuned W =\n", W_full)
print("\nOutput after full update =\n", y_full)

# 4. The central observation behind LoRA

The LoRA hypothesis is roughly:

> The useful change required for adapting a pretrained model may lie in a much lower-dimensional subspace than the enormous size of the original weight matrix suggests.

Instead of learning every element of


$$
\Delta W\in\mathbb{R}^{d_{out}\times d_{in}},
$$


LoRA represents the update as a product of two smaller matrices:


$$
\boxed{\Delta W = BA}
$$


with


$$
A\in\mathbb{R}^{r\times d_{in}}
$$


and


$$
B\in\mathbb{R}^{d_{out}\times r}.
$$


The rank $r$ is chosen much smaller than both $d_{in}$ and $d_{out}$.

So:


$$
r \ll d_{in},d_{out}.
$$


Instead of training


$$
d_{out}d_{in}
$$


numbers, LoRA trains


$$
r d_{in} + d_{out}r
=
r(d_{in}+d_{out}).
$$


That is the mathematical heart of LoRA.

# 5. What does “low rank” actually mean?

Suppose


$$
\Delta W = BA
$$


where $B$ has only $r$ columns and $A$ has only $r$ rows.

Then


$$
\operatorname{rank}(BA)\le r.
$$


So if $r=1$, the learned update can have rank at most 1.

If $r=2$, the update can have rank at most 2.

This constrains the update.

That sounds like a weakness—but in practice it is often exactly what makes the method efficient.

The pretrained matrix $W_0$ remains rich and high-dimensional.

LoRA is **not replacing the original weight with a low-rank matrix**.

It is only saying:


$$
\boxed{\text{new weight}=\text{large pretrained weight}+\text{small low-rank correction}}
$$


or


$$
\boxed{W=W_0+BA}.
$$



# 6. A completely worked rank-1 LoRA example

Return to our $4\times3$ matrix.

Choose LoRA rank


$$
r=1.
$$


Then


$$
A\in\mathbb{R}^{1\times3},
\qquad
B\in\mathbb{R}^{4\times1}.
$$


Let


$$
A=
\begin{bmatrix}
0.2 & -0.1 & 0.3
\end{bmatrix}
$$


and


$$
B=
\begin{bmatrix}
0.5\\
-0.2\\
0.4\\
0.1
\end{bmatrix}.
$$


Now compute


$$
\Delta W = BA.
$$


Because this is an outer-product-like multiplication,


$$
BA=
\begin{bmatrix}
0.5\\
-0.2\\
0.4\\
0.1
\end{bmatrix}
\begin{bmatrix}
0.2&-0.1&0.3
\end{bmatrix}.
$$


Row by row:


$$
0.5A=[0.10,-0.05,0.15]
$$


$$
-0.2A=[-0.04,0.02,-0.06]
$$


$$
0.4A=[0.08,-0.04,0.12]
$$


$$
0.1A=[0.02,-0.01,0.03].
$$


Therefore


$$
\Delta W=
\begin{bmatrix}
0.10 & -0.05 & 0.15\\
-0.04 & 0.02 & -0.06\\
0.08 & -0.04 & 0.12\\
0.02 & -0.01 & 0.03
\end{bmatrix}.
$$



In [ ]:
A = np.array([[0.2, -0.1, 0.3]])       # shape (1, 3)
B = np.array([[ 0.5],
              [-0.2],
              [ 0.4],
              [ 0.1]])                 # shape (4, 1)

delta_W_lora = B @ A

print("A shape:", A.shape)
print("B shape:", B.shape)
print("\nDelta W = B @ A =\n", delta_W_lora)
print("\nRank of Delta W:", np.linalg.matrix_rank(delta_W_lora))

The original full matrix has


$$
4\times3=12
$$


entries.

With rank $r=1$, LoRA trains only


$$
1\times3+4\times1=7
$$


entries.

For this toy matrix that is only a modest saving.

For giant square matrices the saving becomes dramatic.

# 7. Computing the LoRA forward pass two equivalent ways

The effective layer is


$$
y=(W_0+\Delta W)x
$$


and with LoRA


$$
y=(W_0+BA)x.
$$


Distribute the multiplication:


$$
y=W_0x+BAx.
$$


This gives two equivalent interpretations.

### Interpretation A: explicitly form the updated weight


$$
W=W_0+BA
$$


then calculate


$$
y=Wx.
$$


### Interpretation B: keep the original path and add a LoRA branch


$$
y=W_0x+B(Ax).
$$


The second view is extremely useful.

The data flows like this:


$$
x
\rightarrow A
\rightarrow B
\rightarrow \text{add to }W_0x.
$$


Because matrix multiplication is associative,


$$
(BA)x = B(Ax).
$$


We do **not need to explicitly construct $\Delta W$** during every forward pass.

In [ ]:
# Method 1: explicitly make effective W
W_lora = W0 + B @ A
y_method1 = W_lora @ x

# Method 2: base branch + LoRA branch
base_out = W0 @ x
lora_hidden = A @ x
lora_out = B @ lora_hidden
y_method2 = base_out + lora_out

print("Base output W0x:\n", base_out)
print("\nA @ x (low-dimensional bottleneck):\n", lora_hidden)
print("\nB @ (A @ x):\n", lora_out)
print("\nFinal output:\n", y_method2)
print("\nEquivalent?", np.allclose(y_method1, y_method2))

Notice what happens for $r=1$:


$$
Ax
$$


compresses the 3-dimensional input into **one scalar**.

Then $B$ expands that scalar back into four output dimensions.

This is a very small trainable side-path attached to a frozen pretrained transformation.

# 8. The LoRA scaling factor $\alpha/r$

In practice LoRA commonly uses


$$
\boxed{
y=W_0x+\frac{\alpha}{r}BAx
}
$$


rather than simply


$$
W_0x+BAx.
$$


Define


$$
s=\frac{\alpha}{r}.
$$


Then


$$
\Delta W=sBA.
$$


Why?

Because the scale of the adapter contribution can otherwise depend awkwardly on the selected rank.

The hyperparameter $\alpha$ provides a convenient control over LoRA update magnitude.

Example:


$$
r=2,\qquad \alpha=8
$$


gives


$$
s=4.
$$


Different libraries expose this as something like `lora_alpha`.

In [ ]:
alpha = 4
r = 1
scale = alpha / r

scaled_lora_out = (W0 @ x) + scale * (B @ (A @ x))

print("alpha =", alpha)
print("rank =", r)
print("scale = alpha/r =", scale)
print("\nScaled output =\n", scaled_lora_out)

# 9. What is frozen and what is trained?

For a LoRA-adapted matrix:


$$
W_0 \quad \text{is frozen}
$$


while


$$
A,\;B \quad \text{are trainable}.
$$


During backpropagation we conceptually have


$$
\frac{\partial L}{\partial W_0}
$$


available mathematically, but we do not use it to update $W_0$.

Instead, the optimizer updates


$$
A\leftarrow A-\eta\frac{\partial L}{\partial A}
$$


and


$$
B\leftarrow B-\eta\frac{\partial L}{\partial B}.
$$


The pretrained model stays intact.

That has a second important benefit:

> Different tasks can share one common frozen model and store only different LoRA adapters.

# 10. Why one LoRA matrix is commonly initialized to zero

A useful initialization strategy is:

- initialize one factor randomly,
- initialize the other factor to zero.

For example:


$$
A\sim \text{small random values},
\qquad B=0.
$$


Then initially


$$
BA=0.
$$


So at the start of fine-tuning


$$
W=W_0.
$$


The model therefore begins exactly from the pretrained model's behavior.

As training proceeds, $B$ moves away from zero and the low-rank correction begins to matter.

In [ ]:
rng = np.random.default_rng(42)

A_init = rng.normal(0, 0.02, size=(2, 3))
B_init = np.zeros((4, 2))

delta_init = B_init @ A_init

print("Random A:\n", A_init)
print("\nZero B:\n", B_init)
print("\nInitial B @ A:\n", delta_init)
print("\nAll zeros?", np.allclose(delta_init, 0))

# 11. Parameter-count calculation: realistic transformer-sized matrix

Suppose a projection matrix is


$$
W_0\in\mathbb{R}^{4096\times4096}.
$$


Full matrix parameters:


$$
4096\times4096=16,777,216.
$$


Now choose LoRA rank


$$
r=8.
$$


Then


$$
A\in\mathbb{R}^{8\times4096}
$$


contains


$$
8\times4096=32,768
$$


parameters.

And


$$
B\in\mathbb{R}^{4096\times8}
$$


contains another


$$
32,768.
$$


Total LoRA parameters:


$$
65,536.
$$


Fraction trainable:


$$
\frac{65,536}{16,777,216}
\approx0.00390625.
$$


That is


$$
0.390625\%.
$$


So for this matrix, only about **0.39% as many parameters** are trained.

In [ ]:
d_in = 4096
d_out = 4096
r = 8

full_params = d_in * d_out
lora_params = r * d_in + d_out * r

print(f"Full matrix parameters: {full_params:,}")
print(f"LoRA parameters:        {lora_params:,}")
print(f"LoRA/full ratio:        {lora_params/full_params:.6f}")
print(f"Percentage:             {100*lora_params/full_params:.4f}%")
print(f"Reduction factor:       {full_params/lora_params:.1f}x")

# 12. Rank 1, rank 2, rank 8... what changes?

The LoRA rank $r$ controls the capacity of the update.

For a square $d\times d$ matrix:


$$
N_{\text{LoRA}}=2dr.
$$


For $d=4096$:

| Rank $r$ | Trainable LoRA parameters |
|---:|---:|
| 1 | 8,192 |
| 4 | 32,768 |
| 8 | 65,536 |
| 16 | 131,072 |
| 64 | 524,288 |

Higher rank means:

- more trainable capacity,
- more memory,
- more compute,
- potentially better adaptation for difficult tasks.

But rank does **not** necessarily need to be large.

In [ ]:
d = 4096
for rank in [1, 4, 8, 16, 64, 128]:
    n = 2 * d * rank
    pct = 100 * n / (d*d)
    print(f"r={rank:3d}: {n:>10,} parameters  ({pct:.4f}% of full matrix)")

# 13. One layer versus an actual transformer

Everything so far has described **one weight matrix**.

An LLM contains many transformer blocks.

A simplified decoder-only transformer block contains components such as:

### Self-attention projections


$$
Q=XW_Q
$$


$$
K=XW_K
$$


$$
V=XW_V
$$


and an output projection


$$
O=\text{Attention}(Q,K,V)W_O.
$$


### Feed-forward / MLP projections

Depending on the architecture, there may be matrices such as


$$
W_{\text{up}},\qquad
W_{\text{gate}},\qquad
W_{\text{down}}.
$$


Each one of these matrices is a potential LoRA target.

For example, if we LoRA-adapt $W_Q$,


$$
W_Q' = W_Q + B_QA_Q.
$$


If we also adapt $W_V$,


$$
W_V' = W_V + B_VA_V.
$$


These are **separate adapter matrices**.

The $A$ and $B$ for $W_Q$ are not normally shared with $W_V$.

# 14. Gently scaling from one transformer block to many blocks

Suppose the model contains 3 transformer blocks.

To keep notation simple, imagine we apply LoRA only to the query and value projections.

Then block 1 has


$$
W_Q^{(1)} + B_Q^{(1)}A_Q^{(1)}
$$


and


$$
W_V^{(1)} + B_V^{(1)}A_V^{(1)}.
$$


Block 2 has


$$
W_Q^{(2)} + B_Q^{(2)}A_Q^{(2)}
$$


and


$$
W_V^{(2)} + B_V^{(2)}A_V^{(2)}.
$$


Block 3 has


$$
W_Q^{(3)} + B_Q^{(3)}A_Q^{(3)}
$$


and


$$
W_V^{(3)} + B_V^{(3)}A_V^{(3)}.
$$


So six pretrained matrices stay frozen and we train twelve small matrices:

- 6 $A$ matrices
- 6 $B$ matrices.

With 32 transformer blocks and the same two target modules per block, there are


$$
32\times2=64
$$


adapted weight matrices.

Each gets its own LoRA pair.

That is how the one-matrix idea scales to an LLM.

# 15. A parameter-count example across multiple transformer layers

Assume:

- 32 transformer blocks
- model width $d=4096$
- adapt `q_proj` and `v_proj`
- each target matrix is $4096\times4096$
- LoRA rank $r=8$

LoRA parameters per adapted matrix:


$$
2dr=2(4096)(8)=65,536.
$$


Number of adapted matrices:


$$
32\times2=64.
$$


Total trainable LoRA parameters:


$$
64\times65,536=4,194,304.
$$


So only about **4.2 million trainable adapter parameters** are needed for these 64 enormous projection matrices.

The underlying matrices collectively contain


$$
64\times4096^2
=
1,073,741,824
$$


base parameters.

Again, those base parameters are frozen.

In [ ]:
n_layers = 32
targets_per_layer = 2
d = 4096
r = 8

num_target_matrices = n_layers * targets_per_layer
base_params_targeted = num_target_matrices * d * d
lora_trainable = num_target_matrices * (2 * d * r)

print(f"Target matrices:              {num_target_matrices}")
print(f"Frozen base params targeted:  {base_params_targeted:,}")
print(f"Trainable LoRA params:        {lora_trainable:,}")
print(f"Trainable fraction:           {100*lora_trainable/base_params_targeted:.4f}%")

# 16. A tiny multi-layer neural example

Let's make a toy 2-layer network:


$$
h=W_1x
$$


$$
y=W_2h.
$$


Now LoRA-adapt both layers:


$$
h=(W_1+B_1A_1)x
$$


$$
y=(W_2+B_2A_2)h.
$$


The important point is that every layer gets **its own low-rank correction**.

The output of the first adapted layer becomes the input to the second.

So LoRA effects accumulate through the network.

In [ ]:
W1 = np.array([
    [1., 0., 2.],
    [0., 1., 1.],
    [1., 1., 0.]
])

W2 = np.array([
    [ 1.,  2., 0.],
    [-1.,  0., 1.]
])

x2 = np.array([[1.],
               [2.],
               [-1.]])

# rank-1 LoRA for layer 1
A1 = np.array([[0.1, -0.2, 0.3]])
B1 = np.array([[0.4],
               [0.1],
               [-0.2]])

# rank-1 LoRA for layer 2
A2 = np.array([[0.2, 0.1, -0.1]])
B2 = np.array([[0.3],
               [-0.4]])

h_base = W1 @ x2
y_base = W2 @ h_base

h_lora = (W1 + B1 @ A1) @ x2
y_lora = (W2 + B2 @ A2) @ h_lora

print("Base hidden state:\n", h_base)
print("\nLoRA hidden state:\n", h_lora)
print("\nBase output:\n", y_base)
print("\nLoRA output:\n", y_lora)

# 17. LoRA and gradient flow

Now let us derive something important.

For one adapted layer,


$$
y=W_0x+BAx.
$$


Suppose a scalar loss $L$ depends on $y$.

Let


$$
g=\frac{\partial L}{\partial y}.
$$


Then gradients propagate through the LoRA path.

A useful way to view it is:


$$
z=Ax
$$


$$
u=Bz
$$


$$
y=W_0x+u.
$$


Then the backward pass follows the chain rule.

For the $B$ matrix:


$$
\frac{\partial L}{\partial B}
=
g z^T.
$$


Since


$$
z=Ax,
$$


$$
\boxed{
\frac{\partial L}{\partial B}
=
g(Ax)^T
}
$$


For $A$:


$$
\frac{\partial L}{\partial A}
=
(B^Tg)x^T.
$$


Thus the loss supplies learning signals to both low-rank factors.

The frozen base matrix does not need an optimizer update.

# 18. A concrete gradient example

Take rank $r=1$.

Suppose


$$
x=
\begin{bmatrix}
2\\
1
\end{bmatrix},
\qquad
A=
\begin{bmatrix}
0.5&-0.25
\end{bmatrix}
$$


and


$$
B=
\begin{bmatrix}
0.2\\
-0.4
\end{bmatrix}.
$$


First,


$$
z=Ax
=0.5(2)-0.25(1)
=0.75.
$$


Suppose the gradient arriving from later layers is


$$
g=
\begin{bmatrix}
1.5\\
-0.5
\end{bmatrix}.
$$


Then


$$
\frac{\partial L}{\partial B}
=
gz^T.
$$


Because $z$ is scalar,


$$
\frac{\partial L}{\partial B}
=
0.75
\begin{bmatrix}
1.5\\
-0.5
\end{bmatrix}
=
\begin{bmatrix}
1.125\\
-0.375
\end{bmatrix}.
$$


For $A$,


$$
B^Tg
=
[0.2,-0.4]
\begin{bmatrix}
1.5\\
-0.5
\end{bmatrix}
=
0.2(1.5)+(-0.4)(-0.5)
=
0.5.
$$


Therefore


$$
\frac{\partial L}{\partial A}
=
0.5
\begin{bmatrix}
2&1
\end{bmatrix}
=
\begin{bmatrix}
1&0.5
\end{bmatrix}.
$$



In [ ]:
xg = np.array([[2.],
               [1.]])
Ag = np.array([[0.5, -0.25]])
Bg = np.array([[0.2],
               [-0.4]])
g = np.array([[1.5],
              [-0.5]])

z = Ag @ xg
grad_B = g @ z.T
grad_A = (Bg.T @ g) @ xg.T

print("z = A x =", z)
print("\ndL/dB =\n", grad_B)
print("\ndL/dA =\n", grad_A)

# 19. LoRA is not the same as replacing the matrix by a low-rank matrix

This misconception is worth eliminating.

LoRA does **not** say


$$
W\approx BA.
$$


That would throw away the pretrained matrix.

Instead:


$$
\boxed{W=W_0+BA}.
$$


The huge pretrained model is still doing nearly all the representational work.

The low-rank matrices merely steer it toward the new task.

# 20. LoRA adapters can sometimes be merged for inference

After training,


$$
W_{\text{merged}}
=
W_0+\frac{\alpha}{r}BA.
$$


If inference is done in a compatible floating-point setup, this update can often be merged into the base matrix.

Then inference uses a normal matrix multiplication:


$$
y=W_{\text{merged}}x.
$$


This is one reason LoRA can avoid adding a permanent extra sequential module at inference.

However, whether you actually merge depends on:

- model format,
- quantization state,
- serving framework,
- whether you want to switch between adapters dynamically.

For QLoRA-style quantized serving, workflows may keep adapters separate rather than simply merging into a 4-bit base representation.

# 21. Before QLoRA: what is quantization?

Now we introduce the second major idea.

A neural-network weight is usually represented using a numeric format such as FP32, FP16, or BF16.

Very roughly:

- FP32: 32 bits per value
- FP16/BF16: 16 bits per value
- INT8-like storage: 8 bits per value
- 4-bit quantization: around 4 bits per quantized code, plus metadata/scales

If we store a billion numbers:

- 32-bit: about 4 GB just for raw values
- 16-bit: about 2 GB
- 8-bit: about 1 GB
- 4-bit: about 0.5 GB

These are simplified raw-weight calculations; real memory usage includes additional structures and overhead.

Quantization maps many precise floating-point numbers onto a much smaller set of representable values.

# 22. A simple toy quantization example

**This section is deliberately a simplified uniform quantization example. It is not NF4.**

Suppose weights lie roughly in


$$
[-1,1].
$$


Imagine we have only 8 representable levels:


$$
-1.00,\,
-0.714,\,
-0.429,\,
-0.143,\,
0.143,\,
0.429,\,
0.714,\,
1.00.
$$


A weight


$$
0.61
$$


might be stored as the nearest available level:


$$
0.714.
$$


A weight


$$
-0.36
$$


might become


$$
-0.429.
$$


The stored representation is lower precision.

When doing matrix multiplication, software/hardware can dequantize blocks into a compute data type as needed.

In [ ]:
levels = np.linspace(-1, 1, 8)

weights = np.array([-0.93, -0.36, -0.05, 0.21, 0.61, 0.88])

def nearest_level(v, levels):
    return levels[np.argmin(np.abs(levels - v))]

quantized = np.array([nearest_level(v, levels) for v in weights])

print("Levels:")
print(levels)
print("\nOriginal:")
print(weights)
print("\nToy quantized/dequantized:")
print(quantized)
print("\nError:")
print(quantized - weights)

# 23. Why quantization helps memory

Take a 7-billion-parameter model.

Ignoring metadata and runtime overhead:

### 16-bit storage


$$
7\times10^9\times16\text{ bits}
$$


which is approximately


$$
14\text{ GB}
$$


of raw weight storage.

### 4-bit storage


$$
7\times10^9\times4\text{ bits}
$$


which is approximately


$$
3.5\text{ GB}.
$$


This is a huge difference.

But ordinary quantization by itself does not answer the fine-tuning problem.

We still need a practical way to adapt the model.

This leads to QLoRA.

In [ ]:
params = 7_000_000_000

for bits in [32, 16, 8, 4]:
    gb_decimal = params * bits / 8 / 1e9
    print(f"{bits:2d}-bit raw storage for 7B parameters: {gb_decimal:.2f} GB")

# 24. QLoRA in one sentence

QLoRA combines two ideas:

1. **Store the frozen pretrained base model in 4-bit quantized form.**
2. **Train LoRA adapters through that frozen quantized model.**

Conceptually:


$$
W_{0,\text{4bit}}
\quad\text{frozen}
$$


plus trainable higher-precision LoRA parameters


$$
A,\;B.
$$


The effective computation is conceptually something like


$$
y
=
\operatorname{dequant}(W_{0,\text{4bit}})x
+
\frac{\alpha}{r}BAx.
$$


The base matrix remains frozen.

Gradients flow through the computation graph so that $A$ and $B$ can be optimized.

### Crucial correction

QLoRA does **not** fundamentally mean:

> quantize the LoRA adapters to 4 bit.

The key is that the **base pretrained model is 4-bit quantized and frozen**, while the LoRA adapters are the trainable parameters.

# 25. LoRA versus QLoRA: the exact conceptual change

## LoRA

Base weights might be held in FP16/BF16:


$$
W_{0,\text{16bit}}
$$


and


$$
y=W_0x+\frac{\alpha}{r}BAx.
$$


Only $A,B$ are trained.

## QLoRA

Base weights are held in 4-bit quantized storage:


$$
Q_4(W_0).
$$


During computation they are effectively dequantized to a compute representation as needed:


$$
\widetilde W_0
=
D(Q_4(W_0)).
$$


Then


$$
\boxed{
y
=
\widetilde W_0x
+
\frac{\alpha}{r}BAx
}
$$


Again:


$$
A,B\quad\text{are trainable}
$$


while the quantized base model is frozen.

# 26. A concrete toy QLoRA-style forward pass

Let's make a very small matrix:


$$
W_0=
\begin{bmatrix}
0.91&-0.36&0.17\\
-0.72&0.48&0.83\\
0.11&-0.95&0.34
\end{bmatrix}.
$$


We will:

1. quantize it using our simple toy quantizer,
2. freeze that quantized/dequantized approximation,
3. attach rank-1 LoRA,
4. calculate the final output.

Again, the toy quantizer is **not NF4**. It is only to make the mechanics numerically transparent.

In [ ]:
W0_float = np.array([
    [ 0.91, -0.36,  0.17],
    [-0.72,  0.48,  0.83],
    [ 0.11, -0.95,  0.34]
])

levels = np.linspace(-1, 1, 16)  # toy 4-bit-ish 16-level codebook

def toy_quantize_matrix(W, levels):
    out = np.empty_like(W)
    for i in range(W.shape[0]):
        for j in range(W.shape[1]):
            out[i, j] = levels[np.argmin(np.abs(levels - W[i, j]))]
    return out

W0_q_dequant = toy_quantize_matrix(W0_float, levels)

xq = np.array([[1.],
               [2.],
               [-1.]])

Aq = np.array([[0.10, -0.20, 0.05]])
Bq = np.array([[ 0.30],
               [-0.10],
               [ 0.20]])

base_float_out = W0_float @ xq
base_quant_out = W0_q_dequant @ xq
adapter_out = Bq @ (Aq @ xq)
qlora_out = base_quant_out + adapter_out

print("Original float W0:\n", W0_float)
print("\nToy quantized/dequantized W0:\n", W0_q_dequant)
print("\nFloat base output:\n", base_float_out)
print("\nQuantized base output:\n", base_quant_out)
print("\nLoRA correction:\n", adapter_out)
print("\nToy QLoRA-style final output:\n", qlora_out)

This example exposes the central mechanism:


$$
\text{output}
=
\text{frozen quantized-base contribution}
+
\text{trainable low-rank correction}.
$$


Training does **not** normally update the original quantized base codes.

Instead, the optimizer changes the LoRA matrices so that the total network behaves better on the downstream task.

# 27. What does the “Q” in QLoRA buy us?

LoRA already reduces the number of parameters that require gradients and optimizer state.

But the frozen base model still occupies GPU memory.

For a large model, that can be enormous.

QLoRA attacks that remaining cost by compressing the base weights to 4-bit representation.

So:

### LoRA primarily attacks trainable-state cost.

### QLoRA additionally attacks frozen-base storage cost.

This is why QLoRA makes fine-tuning much larger models feasible on limited hardware.

# 28. NF4: NormalFloat 4

The original QLoRA method introduced **NF4 (NormalFloat 4)**.

The key intuition is that neural-network weights are often distributed approximately like a zero-centered bell-shaped distribution rather than uniformly over an interval.

A naive uniform 4-bit codebook spends representational levels evenly across the numeric interval.

NF4 instead uses a non-uniform 4-bit codebook designed for normally distributed weights.

With 4 bits, there are


$$
2^4=16
$$


possible codes.

The important idea is not merely “use 16 equally spaced numbers.”

It is to choose representable values in a way that better matches the statistical distribution of pretrained neural-network weights.

In real QLoRA implementations, quantization is usually performed block-wise with associated scaling information.

# 29. Why we did NOT implement fake NF4 by hand above

It is tempting to say:

> “4-bit quantization means rounding to 16 evenly spaced values.”

That is fine for understanding quantization generally, but it would be wrong to present that as QLoRA's actual NF4 method.

Our toy example demonstrated only the mechanical principle:


$$
\text{float weight}
\rightarrow
\text{compact code}
\rightarrow
\text{dequantized value for compute}.
$$


Real NF4 involves a specialized non-uniform codebook and block-level scaling.

The conceptual QLoRA equation remains:


$$
y
=
D(Q_{\mathrm{NF4}}(W_0))x
+
\frac{\alpha}{r}BAx.
$$



# 30. Double quantization

Quantized weights need quantization metadata such as scale factors.

Those scale factors themselves consume memory.

QLoRA introduces **double quantization**:

> quantize the quantization constants themselves.

Conceptually:


$$
W_0
\overset{\text{quantize}}{\longrightarrow}
(Q,\;c)
$$


where

- $Q$ = low-bit weight codes,
- $c$ = quantization constants/scales.

Double quantization then compresses $c$ too:


$$
c
\overset{\text{quantize again}}{\longrightarrow}
Q_c.
$$


This saves additional memory beyond simply storing the main weights in 4-bit form.

# 31. Paged optimizers

Training memory usage is not perfectly constant.

Long sequences or unusual batches can create temporary memory spikes.

QLoRA introduced **paged optimizers**, leveraging unified-memory style mechanisms to help handle optimizer-state memory spikes.

The high-level purpose is:


$$
\boxed{\text{reduce the chance that temporary training-memory peaks cause out-of-memory failures}}
$$


This is an engineering innovation rather than a change to the core low-rank equation.

# 32. Compute dtype versus storage dtype

This distinction is fundamental.

A QLoRA base matrix might be **stored** in 4-bit form.

That does not mean every arithmetic operation is literally executed as a 4-bit floating-point matrix multiplication.

A typical conceptual pipeline is:


$$
\text{4-bit storage}
\rightarrow
\text{dequantize block}
\rightarrow
\text{BF16/FP16 compute}
\rightarrow
\text{continue forward pass}.
$$


So distinguish:

### Storage precision
How the frozen base weights live in memory.

### Compute precision
The representation used when performing numerical operations.

### Adapter precision
How the trainable LoRA matrices are represented.

These can differ.

# 33. What happens during one QLoRA training step?

Suppose a minibatch contains tokenized training examples.

A simplified training step is:

### Step 1 — token embeddings

Input token IDs become activation vectors.

### Step 2 — enter transformer block 1

For a quantized target matrix $W_Q$:


$$
Q = D(Q_4(W_Q))X + \frac{\alpha}{r}B_QA_QX.
$$


Similar computations happen for any other adapted matrices.

### Step 3 — continue through every transformer block

The model produces logits over the vocabulary.

### Step 4 — compute the loss

For next-token training:


$$
L
=
-\sum_t \log p_\theta(x_t\mid x_{<t}).
$$


### Step 5 — backpropagate

Gradients flow backward through all operations.

The base model participates in the chain rule, but its frozen weights are not optimizer-updated.

### Step 6 — obtain gradients for LoRA parameters

For every adapted module:


$$
\nabla_A L,\qquad \nabla_B L.
$$


### Step 7 — optimizer step

Only the adapter parameters and any explicitly unfrozen auxiliary parameters are updated.

### Step 8 — repeat

After many batches, the LoRA matrices encode the learned adaptation.

# 34. A subtle question: if the base is frozen, why backpropagate through it?

Because later adapter gradients depend on the activations and gradients flowing through earlier and later parts of the model.

“Frozen” means:


$$
\boxed{\text{do not change these parameters}}
$$


It does **not** mean:


$$
\boxed{\text{remove the layer from the differentiation graph}}
$$


We still need the chain rule to carry learning signals through the network.

For example, if an adapter is inserted in transformer block 5, its effect influences blocks 6, 7, 8, and so on.

The final loss must propagate backward through those computations to tell the block-5 adapter how it should change.

# 35. Which transformer matrices should receive LoRA?

There is no universal single choice.

Common target modules in decoder transformers include attention projections such as:

- `q_proj`
- `k_proj`
- `v_proj`
- `o_proj`

and sometimes MLP projections such as:

- `gate_proj`
- `up_proj`
- `down_proj`

A smaller target set saves more parameters.

A broader target set gives the adapter more capacity.

For example:

### Narrow configuration

Adapt only


$$
W_Q,\;W_V.
$$


### Broader configuration

Adapt


$$
W_Q,W_K,W_V,W_O
$$


plus the MLP projection matrices.

The best choice depends on architecture, dataset, task, rank, memory budget, and empirical performance.

# 36. Worked parameter count for a broader transformer configuration

Suppose a model has:

- 32 layers,
- hidden size $4096$,
- four square attention matrices per layer,
- rank $r=16$.

For each $4096\times4096$ target matrix:


$$
N_{\text{LoRA}}
=
4096(16)+16(4096)
=
131,072.
$$


Four attention matrices per layer:


$$
4\times131,072
=
524,288.
$$


Across 32 layers:


$$
32\times524,288
=
16,777,216.
$$


So adapting all four attention projections in all 32 layers gives about **16.8 million trainable LoRA parameters** under these simplified dimensions.

That sounds large, but the corresponding 128 full projection matrices contain:


$$
128\times4096^2
\approx2.147\text{ billion}
$$


base parameters.

Only a small fraction are trainable.

In [ ]:
layers = 32
attn_targets = 4
d = 4096
r = 16

lora_per_matrix = r*d + d*r
lora_total = layers * attn_targets * lora_per_matrix
base_total = layers * attn_targets * d*d

print(f"LoRA params per matrix: {lora_per_matrix:,}")
print(f"Total LoRA params:      {lora_total:,}")
print(f"Corresponding base:     {base_total:,}")
print(f"Trainable fraction:     {100*lora_total/base_total:.4f}%")

# 37. How does LoRA compare with SVD?

LoRA is often explained using the language of low-rank factorization, so beginners sometimes assume training first computes a full $\Delta W$, performs SVD, and truncates it.

That is **not the standard LoRA training procedure**.

Standard LoRA directly parameterizes


$$
\Delta W=BA
$$


and optimizes $A$ and $B$ with gradient descent.

There need not be any explicit full update matrix $\Delta W$ during training.

SVD is useful conceptually because a rank-$r$ matrix can be expressed using low-dimensional factors, but LoRA does not require repeatedly computing SVD of full model updates.

# 38. Why low-rank adaptation can work

There are several useful intuitions.

### 1. Pretraining already did most of the hard work

A pretrained LLM already contains language, syntax, knowledge, patterns, and representations.

Fine-tuning often needs to **steer** behavior rather than relearn intelligence from scratch.

### 2. Task-specific change may occupy a small subspace

Even though the model has billions of parameters, the useful adaptation direction may have much smaller intrinsic dimensionality.

### 3. Redundancy in large networks

Overparameterized networks may allow meaningful behavioral changes through relatively structured parameter perturbations.

### 4. Optimization regularization

Confining the update to a low-rank form restricts the search space, which can sometimes be beneficial when the adaptation dataset is much smaller than the original pretraining corpus.

# 39. Full fine-tuning vs LoRA vs QLoRA

| Property | Full fine-tuning | LoRA | QLoRA |
|---|---|---|---|
| Base model frozen? | Usually no | Yes | Yes |
| Train all original weights? | Yes | No | No |
| Train low-rank adapters? | No | Yes | Yes |
| Base storage typically 4-bit? | Not necessarily | Not necessarily | Yes |
| Optimizer state for full base? | Yes | No | No |
| Adapter parameter count | N/A | Small | Small |
| Main memory advantage | None | Few trainable params | Few trainable params + compressed frozen base |
| Can store one adapter per task? | Not naturally | Yes | Yes |

# 40. Simplified memory intuition

A common beginner mistake is to compare only raw model weights.

Training memory may include:


$$
\text{weights}
+
\text{gradients}
+
\text{optimizer states}
+
\text{activations}
+
\text{temporary buffers}.
$$


Full fine-tuning must maintain optimization information for a huge number of model parameters.

LoRA dramatically reduces the trainable parameter count.

QLoRA also drastically reduces the base-weight storage cost.

However:

> A 4-bit 7B model does **not** imply the entire training job uses only 3.5 GB.

Activations, adapters, temporary dequantized computation, optimizer state, CUDA/runtime overhead, and other tensors still consume memory.

# 41. Toy memory calculator

This calculator only estimates raw parameter storage.

It is intentionally not a full GPU-memory predictor.

In [ ]:
def raw_storage_gb(num_params, bits):
    return num_params * bits / 8 / 1e9

def lora_parameter_count(d_in, d_out, rank, num_matrices=1):
    return num_matrices * rank * (d_in + d_out)

model_params = 7_000_000_000

print("7B base raw storage:")
for bits in [16, 8, 4]:
    print(f"  {bits:2d}-bit: {raw_storage_gb(model_params, bits):.2f} GB")

rank = 16
d = 4096
n_targets = 128
lp = lora_parameter_count(d, d, rank, n_targets)

print(f"\nExample LoRA params: {lp:,}")
print(f"If stored in 16-bit: {raw_storage_gb(lp, 16):.4f} GB")

# 42. What is saved after LoRA/QLoRA fine-tuning?

Instead of saving another entire copy of the base model for each task, you can save the adapter parameters.

For each adapted module, you need matrices like


$$
A^{(\ell,m)},\qquad B^{(\ell,m)}
$$


where

- $\ell$ = transformer layer index,
- $m$ = target module (`q_proj`, `v_proj`, etc.).

You also need configuration metadata such as:

- rank $r$,
- scaling $\alpha$,
- target module names,
- possibly dropout configuration,
- model compatibility information.

The base model can be shared.

Task A loads adapter A.

Task B loads adapter B.

Task C loads adapter C.

# 43. Adapter switching intuition

Imagine one common pretrained model:


$$
W_0.
$$


You train three adapters:


$$
\Delta W_{\text{legal}}
=
B_{\text{legal}}A_{\text{legal}}
$$


$$
\Delta W_{\text{medical}}
=
B_{\text{medical}}A_{\text{medical}}
$$


$$
\Delta W_{\text{SQL}}
=
B_{\text{SQL}}A_{\text{SQL}}.
$$


Then conceptually:


$$
W_{\text{legal}}=W_0+\Delta W_{\text{legal}}
$$


$$
W_{\text{medical}}=W_0+\Delta W_{\text{medical}}
$$


$$
W_{\text{SQL}}=W_0+\Delta W_{\text{SQL}}.
$$


You do not necessarily need three complete copies of the original giant model.

# 44. LoRA dropout

Many implementations optionally apply dropout on the LoRA branch during training.

A schematic version is


$$
y
=
W_0x
+
\frac{\alpha}{r}
BA(\operatorname{Dropout}(x)).
$$


This acts as regularization.

At inference time, dropout is disabled.

`lora_dropout` is therefore a training hyperparameter, not a change to the fundamental definition of LoRA.

# 45. End-to-end pseudocode for LoRA training

```text
load pretrained model

for each target linear layer:
    freeze pretrained W0

    create trainable A
    create trainable B

    forward(x):
        base = W0 @ x
        adapter = (alpha / r) * B @ A @ x
        return base + adapter

for each minibatch:
    logits = model(tokens)
    loss = cross_entropy(logits, targets)

    backpropagate(loss)

    optimizer updates only A and B matrices
```

# 46. End-to-end pseudocode for QLoRA training

```text
load pretrained model

quantize pretrained base weights to 4-bit
freeze all quantized base weights

for each target linear layer:
    create trainable LoRA A and B matrices

for each minibatch:

    for each quantized base layer:
        dequantize blocks as needed for compute
        base_output = base_layer(x)

        if layer has LoRA:
            adapter_output = (alpha / r) * B @ A @ x
            output = base_output + adapter_output

    logits = model(tokens)
    loss = cross_entropy(logits, target_tokens)

    backpropagate(loss)

    optimizer updates only trainable adapters
```

Real systems use highly optimized kernels rather than literally materializing every full dequantized matrix in a naive way.

# 47. A minimal PyTorch-style LoRA layer from scratch

The next cell shows the structure without relying on a PEFT library.

It is educational code rather than a production implementation.

In [ ]:
# This cell requires PyTorch if you want to execute it.

import torch
import torch.nn as nn

class LoRALinear(nn.Module):
    def __init__(self, base_linear: nn.Linear, rank=4, alpha=8):
        super().__init__()

        self.base = base_linear
        self.rank = rank
        self.alpha = alpha
        self.scaling = alpha / rank

        # Freeze the pretrained base layer
        for p in self.base.parameters():
            p.requires_grad = False

        d_in = base_linear.in_features
        d_out = base_linear.out_features

        # A: rank x d_in
        self.A = nn.Parameter(torch.empty(rank, d_in))

        # B: d_out x rank
        self.B = nn.Parameter(torch.zeros(d_out, rank))

        nn.init.normal_(self.A, mean=0.0, std=0.02)

    def forward(self, x):
        # nn.Linear internally applies x @ W^T
        base_out = self.base(x)

        # x: [..., d_in]
        # A.T: [d_in, rank]
        # B.T: [rank, d_out]
        lora_out = (x @ self.A.T) @ self.B.T

        return base_out + self.scaling * lora_out

# 48. Check which parameters are trainable

In a proper LoRA layer, the original linear-layer parameters should be frozen while $A$ and $B$ remain trainable.

In [ ]:
base = nn.Linear(6, 5, bias=False)
layer = LoRALinear(base, rank=2, alpha=4)

for name, p in layer.named_parameters():
    print(f"{name:15s} shape={tuple(p.shape)!s:12s} requires_grad={p.requires_grad}")

Expected pattern:

```text
A             trainable
B             trainable
base.weight   frozen
```

This tiny example captures the same parameter-freezing logic used in large LoRA fine-tuning systems.

# 49. Verify the adapter starts as zero

Because $B=0$,


$$
BA=0.
$$


So initially the LoRA-wrapped layer should produce exactly the same output as the original base layer.

In [ ]:
torch.manual_seed(0)

base = nn.Linear(3, 4, bias=False)
layer = LoRALinear(base, rank=2, alpha=4)

xt = torch.randn(5, 3)

with torch.no_grad():
    base_y = base(xt)
    lora_y = layer(xt)

print("Maximum difference at initialization:",
      (base_y - lora_y).abs().max().item())

# 50. Demonstrate that gradients reach only the adapter

Now we will perform one toy backward pass.

In [ ]:
torch.manual_seed(1)

base = nn.Linear(3, 2, bias=False)
layer = LoRALinear(base, rank=1, alpha=1)

x_train = torch.tensor([[1.0, 2.0, -1.0]])
target = torch.tensor([[0.5, -0.25]])

pred = layer(x_train)
loss = ((pred - target) ** 2).mean()
loss.backward()

for name, p in layer.named_parameters():
    grad_status = None if p.grad is None else p.grad.detach().clone()
    print(f"{name}:\n  requires_grad={p.requires_grad}\n  grad={grad_status}\n")

A subtle detail may appear here:

Because $B$ starts at zero, the first gradient for $A$ can be zero.

Why?

Recall


$$
\frac{\partial L}{\partial A}
=
(B^Tg)x^T.
$$


If


$$
B=0,
$$


then


$$
\frac{\partial L}{\partial A}=0.
$$


But


$$
\frac{\partial L}{\partial B}
=
g(Ax)^T
$$


can be nonzero because $A$ was randomly initialized.

So the first update moves $B$.

After that, $A$ can also receive nonzero gradients.

This is a beautiful concrete example of why the common zero/random initialization works.

# 51. Two optimization steps to see both factors move

Let's explicitly demonstrate the effect.

In [ ]:
torch.manual_seed(2)

base = nn.Linear(3, 2, bias=False)
layer = LoRALinear(base, rank=1, alpha=1)

optimizer = torch.optim.SGD([layer.A, layer.B], lr=0.1)

x_train = torch.tensor([[1.0, 2.0, -1.0]])
target = torch.tensor([[0.5, -0.25]])

for step in range(3):
    optimizer.zero_grad()

    pred = layer(x_train)
    loss = ((pred - target) ** 2).mean()
    loss.backward()

    print(f"Step {step}")
    print("  loss =", loss.item())
    print("  ||grad A|| =", layer.A.grad.norm().item())
    print("  ||grad B|| =", layer.B.grad.norm().item())

    optimizer.step()

# 52. Where Hugging Face PEFT-style code fits conceptually

In practice you usually do **not** manually rewrite every linear layer.

A parameter-efficient fine-tuning library:

1. finds target modules,
2. wraps or replaces them with LoRA-aware modules,
3. freezes the base parameters,
4. creates trainable $A,B$ matrices,
5. exposes only the adapter parameters to the optimizer,
6. saves compact adapter checkpoints.

A conceptual configuration often looks like:

```python
rank = 16
alpha = 32
target_modules = [
    "q_proj",
    "k_proj",
    "v_proj",
    "o_proj"
]
```

The exact names depend on the model architecture.

# 53. Conceptual QLoRA configuration

A typical QLoRA setup has two independent configuration groups.

### Quantization configuration

Conceptually:

```python
load_in_4bit = True
quant_type = "nf4"
use_double_quant = True
compute_dtype = bfloat16
```

### LoRA configuration

Conceptually:

```python
r = 16
lora_alpha = 32
target_modules = [...]
lora_dropout = ...
```

The first group controls the frozen base model representation.

The second group controls the trainable adapters.

That separation is important.

# 54. Why QLoRA can still learn if the base matrix is approximate

Quantization introduces error:


$$
\widetilde W_0
=
W_0+E_q
$$


where


$$
E_q
$$


is quantization error.

The adapted QLoRA layer is roughly


$$
W_{\text{effective}}
=
W_0+E_q+BA.
$$


Training can choose $BA$ so the complete model performs well on the downstream objective.

You should **not** interpret this as the adapter merely learning to cancel quantization error.

The adapter is learning the downstream task.

But the optimization naturally occurs in the presence of the quantized base representation.

# 55. Sequence-level view inside an LLM

Until now $x$ looked like one vector.

Transformers process many token positions at once.

If sequence length is $T$, activations might be represented as


$$
X\in\mathbb{R}^{T\times d_{in}}.
$$


Using row-vector batch notation, a linear layer computes


$$
Y=XW_0^T.
$$


The LoRA version becomes


$$
Y
=
XW_0^T
+
\frac{\alpha}{r}
X A^T B^T.
$$


Shape check:


$$
X:
T\times d_{in}
$$


$$
A^T:
d_{in}\times r
$$


therefore


$$
XA^T:
T\times r.
$$


Then


$$
B^T:
r\times d_{out}
$$


so


$$
XA^TB^T:
T\times d_{out}.
$$


It matches the base output shape perfectly.

# 56. Add batch size

With batch size $N$,


$$
X\in\mathbb{R}^{N\times T\times d_{in}}.
$$


The same linear transformation is applied over the final dimension.

Conceptually:


$$
Y[n,t,:]
=
W_0X[n,t,:]
+
\frac{\alpha}{r}BA X[n,t,:].
$$


Modern tensor libraries vectorize this across all batches and token positions.

# 57. Why LoRA reduces optimizer memory so strongly

Take Adam-like optimization.

For every trainable parameter, an optimizer may keep extra running statistics, commonly first and second moment estimates.

So one trainable scalar may imply memory for:

- parameter value,
- gradient,
- first moment,
- second moment,
- and sometimes master precision copies or other implementation-specific state.

If 7 billion base parameters are frozen, they do not require the same gradient and optimizer state as full fine-tuning.

Instead, optimizer state exists only for the much smaller adapter parameter set.

This is a major source of LoRA's training-memory savings.

# 58. What LoRA does NOT remove

LoRA is highly memory efficient, but it does not make training free.

You still need memory for:

- activations,
- attention intermediates,
- KV-like temporary tensors used during the forward/backward computation,
- gradients for trainable adapters,
- adapter optimizer state,
- model weights,
- runtime/kernel workspaces.

Techniques such as:

- gradient checkpointing,
- smaller micro-batches,
- shorter sequence lengths,
- mixed precision,
- efficient attention kernels,

may still be important.

# 59. Common misconception: “rank 8 means only 8 neurons are trained”

No.

Rank $r=8$ means the update to a target weight matrix is factorized through an 8-dimensional intermediate space:


$$
d_{in}
\rightarrow
8
\rightarrow
d_{out}.
$$


It does not mean the transformer has only eight active neurons.

The original full layer still operates.

LoRA adds a rank-constrained correction to it.

# 60. Common misconception: “The pretrained weights are converted into A and B”

No.

The pretrained matrix


$$
W_0
$$


remains.

The new matrices


$$
A,B
$$


are additional parameters.

The effective update is


$$
W_0+BA.
$$


LoRA is additive adaptation, not a replacement decomposition of the original pretrained matrix.

# 61. Common misconception: “QLoRA trains the 4-bit base weights”

In standard QLoRA:


$$
Q_4(W_0)
$$


is frozen.

The trainable objects are the adapter parameters.

Gradients pass through the computations involving the quantized/dequantized base model, but the optimizer does not update those frozen base weights.

# 62. Common misconception: “4-bit training means everything is 4-bit”

No.

A QLoRA system can simultaneously use:

- 4-bit base-weight storage,
- BF16/FP16 compute,
- higher-precision adapter parameters,
- optimizer states in another representation.

“4-bit” refers primarily to the quantized frozen base weights, not every tensor in the training process.

# 63. Common misconception: “LoRA has zero inference cost in every deployment”

The original LoRA paper emphasizes that the learned update can be merged into the pretrained weight in suitable settings:


$$
W_{\text{merged}}
=
W_0+\frac{\alpha}{r}BA.
$$


Then there is no separate adapter path at inference.

But practical serving systems may intentionally keep adapters separate because they want:

- dynamic adapter switching,
- multiple tenants,
- quantized base models,
- modular deployment.

So the exact inference overhead depends on the serving setup.

# 64. Hyperparameters you should understand

## Rank $r$

Controls adapter capacity.

Higher $r$:

- more trainable parameters,
- more memory,
- more expressive update.

## Alpha $\alpha$

Controls scaling:


$$
s=\alpha/r.
$$


## Target modules

Controls where adapters are inserted.

## LoRA dropout

Regularization on the adapter path.

## Learning rate

LoRA often tolerates or uses learning rates different from full fine-tuning because only a small set of newly introduced parameters is optimized.

## Batch size / gradient accumulation

Controls optimization and memory tradeoffs.

## Sequence length

A major contributor to activation memory.

# 65. A compact mathematical summary

## Full fine-tuning


$$
W=W_0+\Delta W
$$


where


$$
\Delta W\in\mathbb{R}^{d_{out}\times d_{in}}
$$


is unconstrained and potentially every element is trained.

---

## LoRA


$$
W=W_0+\frac{\alpha}{r}BA
$$


with


$$
A\in\mathbb{R}^{r\times d_{in}}
$$


and


$$
B\in\mathbb{R}^{d_{out}\times r}.
$$


Frozen:


$$
W_0.
$$


Trainable:


$$
A,B.
$$


Parameter count:


$$
r(d_{in}+d_{out}).
$$


---

## QLoRA

Store the frozen base approximately as


$$
Q_4(W_0).
$$


Use a dequantized compute view:


$$
\widetilde W_0=D(Q_4(W_0)).
$$


Then


$$
y
=
\widetilde W_0x
+
\frac{\alpha}{r}BAx.
$$


Train:


$$
A,B.
$$


Freeze:


$$
Q_4(W_0).
$$



# 66. Mental model to remember

If you remember only one picture, remember this.

## Full fine-tuning

```text
         every giant base weight changes
x  ─────────────────────────────────────────► y
```

## LoRA

```text
          frozen giant pretrained layer
x  ─────────────────────────────────────────► +
 \                                            │
  \→ small trainable A → small trainable B ───┘
```

## QLoRA

```text
       frozen giant pretrained layer
       STORED IN 4-BIT QUANTIZED FORM
x  ─────────────────────────────────────────► +
 \                                            │
  \→ small trainable A → small trainable B ───┘
```

LoRA reduces how much you **train**.

QLoRA also reduces how much memory the frozen base model uses.

# 67. Final worked comparison

Suppose one target matrix has


$$
d_{in}=d_{out}=4096.
$$


Then


$$
N_{\text{full}}
=
4096^2
=
16,777,216.
$$


With rank


$$
r=16,
$$


LoRA trains


$$
N_{\text{LoRA}}
=
16(4096+4096)
=
131,072.
$$


So the trainable fraction is


$$
\frac{131,072}{16,777,216}
=
0.0078125
=
0.78125\%.
$$


LoRA keeps the base model at its original storage precision.

QLoRA keeps the **same low-rank training strategy**, but stores the frozen base weights in 4-bit quantized form.

Therefore:


$$
\boxed{\text{LoRA = low-rank trainable update}}
$$


$$
\boxed{\text{QLoRA = LoRA + 4-bit frozen base model}}
$$



# 68. Exercises

Try these by hand before running code.

### Exercise 1

Let


$$
A=
\begin{bmatrix}
1&2&-1
\end{bmatrix}
$$


and


$$
B=
\begin{bmatrix}
2\\
-1
\end{bmatrix}.
$$


Calculate


$$
BA.
$$


What is its rank?

---

### Exercise 2

For


$$
d_{in}=3072,
\quad
d_{out}=3072,
\quad
r=8,
$$


calculate:

1. full matrix parameter count,
2. LoRA parameter count,
3. percentage trainable.

---

### Exercise 3

A model has 24 transformer layers.

You LoRA-adapt four matrices per layer.

Each matrix has shape


$$
4096\times4096
$$


and $r=8$.

Calculate total LoRA trainable parameter count.

---

### Exercise 4

Explain in one sentence why QLoRA still needs BF16/FP16-like arithmetic even though its base weights may be stored in 4-bit format.

---

### Exercise 5

Why can the gradient for $A$ be zero on the very first optimization step if $B$ is initialized to zero?

In [ ]:
# Exercise helper

def lora_stats(d_in, d_out, rank, num_matrices=1):
    full = num_matrices * d_in * d_out
    lora = num_matrices * rank * (d_in + d_out)
    return {
        "full_parameters": full,
        "lora_parameters": lora,
        "percentage": 100 * lora / full
    }

print(lora_stats(3072, 3072, 8))

print(lora_stats(
    d_in=4096,
    d_out=4096,
    rank=8,
    num_matrices=24*4
))

# 69. Exercise solutions

## Exercise 1


$$
BA=
\begin{bmatrix}
2\\
-1
\end{bmatrix}
\begin{bmatrix}
1&2&-1
\end{bmatrix}
=
\begin{bmatrix}
2&4&-2\\
-1&-2&1
\end{bmatrix}.
$$


The second row is $-\tfrac12$ times the first, so the matrix has rank 1.

---

## Exercise 2

Full:


$$
3072^2
=
9,437,184.
$$


LoRA:


$$
8(3072+3072)
=
49,152.
$$


Fraction:


$$
49,152/9,437,184
\approx0.0052083.
$$


Percentage:


$$
0.52083\%.
$$


---

## Exercise 3

Number of target matrices:


$$
24\times4=96.
$$


LoRA parameters per matrix:


$$
8(4096+4096)=65,536.
$$


Total:


$$
96\times65,536=6,291,456.
$$


---

## Exercise 4

The 4-bit representation is primarily a compact storage format; blocks are dequantized into a suitable compute representation for numerical operations.

---

## Exercise 5

Because


$$
\frac{\partial L}{\partial A}
=
(B^Tg)x^T,
$$


and if $B=0$, the first factor is zero.

# 70. References

The foundational papers are:

1. **LoRA: Low-Rank Adaptation of Large Language Models**  
   Edward J. Hu et al., 2021.

2. **QLoRA: Efficient Finetuning of Quantized LLMs**  
   Tim Dettmers et al., 2023.

The LoRA paper introduced the idea of freezing pretrained weights while learning low-rank update matrices.

The QLoRA paper combined LoRA with a frozen 4-bit quantized pretrained model and introduced techniques including NF4, double quantization, and paged optimizers.

For production use, also consult the current documentation for the model framework and PEFT/quantization libraries you actually use, because APIs and supported kernels evolve.

# 71. Where to go next

A natural continuation after this notebook is:

1. derive transformer self-attention in full,
2. inspect actual $W_Q,W_K,W_V,W_O$ matrix shapes,
3. inject LoRA manually into a tiny transformer,
4. train it on a tiny synthetic dataset,
5. compare:
   - full fine-tuning,
   - LoRA,
   - frozen model,
6. then repeat with a real small pretrained language model,
7. finally move to a genuine 4-bit QLoRA stack.

That sequence turns the equations here into complete implementation-level understanding.